Ten skrypt wykorzystuje dwa lokalne modele językowe  do tworzenia, analizowania i ulepszania żartów typu "dad joke" poprzez współpracę trzech wyspecjalizowanych agentów: kreatora, parsera, i sędziego. System generuje żart, szczegółowo go analizuje pod kątem ośmiu kryteriów oceny, prezentuje wyniki w estetycznej tabeli, a następnie tworzy ulepszoną wersję na podstawie wcześniejszej krytyki.

In [ ]:
# TERMINAL -> alternatywnie https://github.com/InfuseAI/colab-xterm

# curl -fsSL https://ollama.com/install.sh | PATH="/sbin:/usr/sbin:$PATH" sh

# ollama serve &

#ollama pull deepseek-r1:14b

#ollama pull qwen2.5

Ten zestaw poleceń służy do uruchomienia i skonfigurowania Ollamy, narzędzia do lokalnego uruchamiania dużych modeli językowych (LLM).

Pierwsze polecenie pobiera skrypt instalacyjny Ollamy za pomocą `curl` i wykonuje go przy użyciu `sh`.  Ustawienie zmiennej środowiskowej `PATH` zapewnia, że programy zainstalowane przez ten skrypt będą dostępne w systemie.

Drugie polecenie uruchamia serwer Ollamy w tle (`&`). Serwer jest niezbędny do działania modeli językowych pobranych za pomocą Ollamy.

Kolejne dwa polecenia służą do pobrania konkretnych modeli językowych: `deepseek-r1:14b` oraz `qwen2.5`.  Ollama automatycznie pobierze i skonfiguruje te modele, aby były gotowe do użycia.


# Setup

In [1]:
!pip install -q pydantic_ai pydantic rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.6/156.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.5/301.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Te polecenia instalują trzy biblioteki Pythona za pomocą menedżera pakietów `pip`. Flaga `-q` oznacza tryb cichy, co minimalizuje ilość wyświetlanych informacji podczas instalacji.

*   **pydantic_ai:** Biblioteka ta rozszerza możliwości Pydantic (biblioteki do walidacji danych) o integrację z modelami językowymi AI. Umożliwia ona wykorzystanie modeli AI do generowania i przetwarzania danych zgodnych ze schematami Pydantic.
*   **pydantic:** Jest to biblioteka służąca do definiowania typów danych i walidacji danych w Pythonie. Pomaga zapewnić, że dane spełniają określone kryteria przed ich użyciem.
*   **rich:** Biblioteka ta dodaje kolorowe i stylowe formatowanie tekstu do konsoli Pythona. Ułatwia to czytanie i interpretację wyników oraz komunikatów w terminalu.

In [30]:
class CFG:
  model1 = 'deepseek-r1:14b'
  model2 = 'qwen2.5'

Ten kod definiuje klasę o nazwie `CFG` (skrót od Configuration, czyli konfiguracja). Klasa ta służy do przechowywania ustawień lub parametrów programu w jednym miejscu.

Wewnątrz klasy zdefiniowane są dwie zmienne:

*   **model1:** Przypisana jest wartość `'deepseek-r1:14b'`. Ta wartość reprezentuje nazwę pierwszego modelu językowego, który ma być używany.
*   **model2:** Przypisana jest wartość `'qwen2.5'`.  Ta wartość reprezentuje nazwę drugiego modelu językowego, który ma być używany.

Dzięki temu kodowi można łatwo uzyskać dostęp do tych nazw modeli w programie poprzez odwołanie się do nich za pomocą `CFG.model1` i `CFG.model2`. Ułatwia to zarządzanie konfiguracją programu i zmianę używanych modeli bez modyfikacji kodu głównego.

In [8]:
# Standard library imports
import asyncio
from typing import List

# Third-party imports
from pydantic import BaseModel, Field, confloat
from rich import box
from rich.console import Console
from rich.table import Table

# Pydantic AI imports
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

Ten blok kodu zawiera instrukcje `import`, które wczytują niezbędne biblioteki i moduły do programu. Można je podzielić na trzy kategorie:

*   **Standard library imports:** Zawiera importy z bibliotek standardowych Pythona, takich jak `asyncio` (do programowania asynchronicznego) oraz `typing` (do adnotacji typów).
*   **Third-party imports:** Zawiera importy z zewnętrznych bibliotek zainstalowanych za pomocą `pip`, takich jak:
    *   `pydantic`: Do definiowania modeli danych i walidacji. Używane są klasy `BaseModel`, `Field` oraz `confloat`.
    *   `rich`: Do tworzenia estetycznych interfejsów w konsoli, z wykorzystaniem klas `Console` i `Table`.  Używana jest również stała `box` do definiowania stylów ramek tabel.
*   **Pydantic AI imports:** Zawiera importy specyficzne dla biblioteki `pydantic_ai`, które umożliwiają integrację modeli językowych z Pydantic. Importowane są:
    *   `Agent`: Klasa reprezentująca agenta AI, który wykorzystuje model językowy do wykonywania zadań.
    *   `RunContext`:  Klasa przechowująca kontekst wykonania agenta.
    *   `OpenAIModel`: Model reprezentujący modele OpenAI dostępne w `pydantic_ai`.
    *   `OpenAIProvider`: Dostawca usług OpenAI dla `pydantic_ai`, umożliwiający komunikację z API OpenAI.

Wszystkie te importy przygotowują środowisko do działania programu, który  wykorzystuje modele językowe AI (takie jak OpenAI) do przetwarzania danych i generowania odpowiedzi, a następnie prezentuje wyniki w estetyczny sposób w konsoli.

In [31]:

# DeepSeekR1
deepseek_reasoner_model = OpenAIModel(
    CFG.model1,
    provider=OpenAIProvider(base_url='http://localhost:11434/v1')
)

deepseek_parser_model = OpenAIModel(
    CFG.model2,
    provider=OpenAIProvider(base_url='http://localhost:11434/v1')
)


Ten kod tworzy dwie instancje klasy `OpenAIModel`, reprezentujące dwa różne modele językowe uruchomione lokalnie przez Ollamę.

*   **deepseek\_reasoner\_model:** Ta zmienna przechowuje obiekt modelu, który będzie używany do rozumowania lub logicznego myślenia. Jest on skonfigurowany do korzystania z modelu określonego w `CFG.model1`, czyli `'deepseek-r1:14b'`.  `OpenAIProvider` jest ustawiony na komunikację z serwerem Ollamy działającym lokalnie pod adresem `http://localhost:11434/v1`.

*   **deepseek\_parser\_model:** Ta zmienna przechowuje obiekt modelu, który będzie używany do parsowania lub analizy tekstu. Jest on skonfigurowany do korzystania z modelu określonego w `CFG.model2`, czyli `'qwen2.5'`. Podobnie jak poprzednio, `OpenAIProvider` jest ustawiony na komunikację z lokalnym serwerem Ollamy pod adresem `http://localhost:11434/v1`.

W skrócie, kod ten inicjalizuje dwa modele językowe – DeepSeek-R1 i Qwen2.5 – które będą dostępne do użycia w programie poprzez interfejs udostępniany przez bibliotekę `pydantic_ai`. Oba modele komunikują się z serwerem Ollamy działającym lokalnie.

# Funkcje

In [12]:

# Define the Joke model
class Joke(BaseModel):
    setup: str = Field(..., description="The setup part of the joke")
    punchline: str = Field(..., description="The punchline of the joke")
    category: str = Field(..., description="Category of the joke (e.g., 'pun', 'wordplay', 'dad joke')")
    thinking: str = Field(..., description="the thought process and planning")


Ten kod definiuje model danych o nazwie `Joke` przy użyciu biblioteki Pydantic. Model ten reprezentuje żart i określa jego strukturę oraz wymagane pola.

*   **class Joke(BaseModel):** Definiuje klasę `Joke`, która dziedziczy po `BaseModel` z Pydantic, co oznacza, że będzie to model danych z walidacją.
*   **setup: str = Field(..., description="The setup part of the joke")**:  Definiuje pole `setup` jako ciąg znaków (`str`). `Field(...)` oznacza, że to pole jest wymagane (nie może być puste). `description` dodaje opis pola, który może być używany przez modele językowe AI do zrozumienia jego przeznaczenia. To pole będzie zawierało początek żartu.
*   **punchline: str = Field(..., description="The punchline of the joke")**: Definiuje pole `punchline` jako ciąg znaków (`str`), które również jest wymagane i opisane.  To pole będzie zawierało puentę żartu.
*   **category: str = Field(..., description="Category of the joke (e.g., 'pun', 'wordplay', 'dad joke')")**: Definiuje pole `category` jako ciąg znaków (`str`), które również jest wymagane i opisane. To pole będzie zawierało kategorię żartu, np. "dowcip słowny", "gra słów" lub "żart taty".
*   **thinking: str = Field(..., description="the thought process and planning")**: Definiuje pole `thinking` jako ciąg znaków (`str`), które również jest wymagane i opisane. To pole będzie zawierało informacje o procesie myślowym i planowaniu, jakie doprowadziły do stworzenia żartu.

Ten model danych służy do strukturyzowania informacji o żartach w programie i może być używany do walidacji danych wejściowych lub wyjściowych z modeli językowych AI. Opisy pól są szczególnie ważne dla `pydantic_ai`, ponieważ pomagają agentowi AI zrozumieć, jakie dane powinny być generowane.

In [13]:

class JokeAssessment(BaseModel):
    originality: confloat(ge=0, le=10) = Field(..., description="Rating of joke originality from 0-10")
    humor_level: confloat(ge=0, le=10) = Field(..., description="Rating of humor level from 0-10")
    family_friendly: bool = Field(..., description="Whether the joke is family-friendly")
    wordplay_quality: confloat(ge=0, le=10) = Field(..., description="Rating of wordplay quality from 0-10")
    delivery_structure: confloat(ge=0, le=10) = Field(..., description="Rating of setup-punchline structure from 0-10")
    strengths: List[str] = Field(..., description="List of joke's strong points")
    improvement_suggestions: List[str] = Field(..., description="Think step buy step and provide a very extensive List of very specific suggestions for improvement")
    overall_score: confloat(ge=0, le=10) = Field(..., description="Overall joke quality score from 0-10")

Ten kod definiuje model danych o nazwie `JokeAssessment` przy użyciu biblioteki Pydantic. Model ten służy do oceny żartu i zawiera różne kryteria oraz pola, które opisują jakość żartu.

*   **class JokeAssessment(BaseModel):**: Definiuje klasę `JokeAssessment`, która dziedziczy po `BaseModel` z Pydantic, co oznacza, że będzie to model danych z walidacją.
*   **originality: confloat(ge=0, le=10) = Field(..., description="Rating of joke originality from 0-10")**: Definiuje pole `originality` jako liczbę zmiennoprzecinkową (`confloat`) w zakresie od 0 do 10.  `ge=0` i `le=10` oznaczają, że wartość musi być większa lub równa 0 i mniejsza lub równa 10. To pole reprezentuje ocenę oryginalności żartu.
*   **humor\_level: confloat(ge=0, le=10) = Field(..., description="Rating of humor level from 0-10")**: Definiuje pole `humor_level` jako liczbę zmiennoprzecinkową w zakresie od 0 do 10. Reprezentuje ocenę poziomu humoru żartu.
*   **family\_friendly: bool = Field(..., description="Whether the joke is family-friendly")**: Definiuje pole `family_friendly` jako wartość logiczną (`bool`). Określa, czy żart jest odpowiedni dla całej rodziny.
*   **wordplay\_quality: confloat(ge=0, le=10) = Field(..., description="Rating of wordplay quality from 0-10")**: Definiuje pole `wordplay_quality` jako liczbę zmiennoprzecinkową w zakresie od 0 do 10. Reprezentuje ocenę jakości gry słów w żarcie.
*   **delivery\_structure: confloat(ge=0, le=10) = Field(..., description="Rating of setup-punchline structure from 0-10")**: Definiuje pole `delivery_structure` jako liczbę zmiennoprzecinkową w zakresie od 0 do 10. Reprezentuje ocenę struktury żartu (początek i puenta).
*   **strengths: List[str] = Field(..., description="List of joke's strong points")**: Definiuje pole `strengths` jako listę ciągów znaków (`List[str]`). Zawiera listę mocnych stron żartu.
*   **improvement\_suggestions: List[str] = Field(..., description="Think step buy step and provide a very extensive List of very specific suggestions for improvement")**: Definiuje pole `improvement_suggestions` jako listę ciągów znaków (`List[str]`). Zawiera szczegółowe sugestie dotyczące poprawy żartu. Opis wyraźnie nakazuje modelowi AI, aby myślał krok po kroku i dostarczył obszerną listę konkretnych propozycji.
*   **overall\_score: confloat(ge=0, le=10) = Field(..., description="Overall joke quality score from 0-10")**: Definiuje pole `overall_score` jako liczbę zmiennoprzecinkową w zakresie od 0 do 10. Reprezentuje ogólną ocenę jakości żartu.

Ten model danych służy do strukturyzowania oceny żartów i może być używany przez modele językowe AI do analizy i krytyki żartów, a także do generowania sugestii dotyczących ich poprawy.

# Agenci

In [34]:

joke_creator = Agent(
    model = deepseek_reasoner_model,
    result_type=str,
    deps_type=str,
    system_prompt=(
        "You are a joke creator that specializes in creating funny, family-friendly dad jokes. "
        "Your output MUST follow this EXACT format with EXACT field names:\n\n"
        "Thinking: \n<think>\n[your detailed thought process without changes]\n</think>\n\n"
        "Setup: [setup text]\n"
        "Punchline: [punchline text]\n"
        "Category: [category]\n"
    ),
)

Ten kod tworzy instancję klasy `Agent` z biblioteki `pydantic_ai`, która będzie pełniła rolę kreatora żartów.

*   **joke\_creator = Agent(...):**: Tworzy obiekt o nazwie `joke_creator` będący agentem AI.
*   **model = deepseek\_reasoner\_model:** Określa, który model językowy ma być używany przez agenta. W tym przypadku jest to `deepseek_reasoner_model`, czyli wcześniej zdefiniowany model DeepSeek-R1 skonfigurowany do rozumowania.
*   **result\_type=str:** Definiuje typ danych oczekiwanego wyniku działania agenta jako ciąg znaków (`str`).
*   **deps\_type=str:** Określa typ danych zależności (dependencies) agenta, również jako ciąg znaków.
*   **system\_prompt=(...):**:  Definiuje instrukcję systemową dla agenta. Instrukcja ta określa rolę agenta i format wyjściowy.
    *   Agent ma być kreatorem żartów specjalizującym się w zabawnych, przyjaznych rodzinie dowcipach typu "tata joke".
    *   Wyjście agenta *musi* ściśle przestrzegać określonego formatu z dokładnymi nazwami pól:
        *   **Thinking:** Sekcja zawierająca szczegółowy proces myślowy agenta, umieszczona w tagach
        *   **Setup:** Tekst wprowadzenia do żartu.
        *   **Punchline:** Tekst puenty żartu.
        *   **Category:** Kategoria żartu.

Ten kod konfiguruje agenta AI, który będzie generował żarty w określonym formacie, wykorzystując model DeepSeek-R1 do rozumowania i tworzenia treści.  Szczególny nacisk położono na ścisłe przestrzeganie formatu wyjściowego, co jest istotne dla dalszego przetwarzania danych przez program.

In [35]:

joke_parser = Agent(
    model=deepseek_parser_model,
    result_type=Joke,
    deps_type=str,  # Takes raw text as input
    system_prompt=(
        "You are a joke parser that extracts components from the input text to create a Joke object. "
        "Your ONLY task is to find and extract these components from the input text:\n"
        "1. The setup (after 'Setup:')\n"
        "2. The punchline (after 'Punchline:')\n"
        "3. The category (after 'Category:')\n"
        "4. The thinking (between <think> tags)\n\n"
        "Return the components in this format:\n"
        "setup=[text after Setup:]\n"
        "punchline=[text after Punchline:]\n"
        "category=[text after Category:]\n"
        "thinking=[text between <think> tags]\n\n"
        "DO NOT generate new content or modify the text in any way.\n"
        "DO NOT include any other text in your response."
    ),
)

Ten kod tworzy instancję klasy `Agent` z biblioteki `pydantic_ai`, która będzie pełniła rolę parsera żartów.

*   **joke\_parser = Agent(...):**: Tworzy obiekt o nazwie `joke_parser` będący agentem AI.
*   **model=deepseek\_parser\_model:** Określa, który model językowy ma być używany przez agenta. W tym przypadku jest to `deepseek_parser_model`, czyli wcześniej zdefiniowany model Qwen2.5 skonfigurowany do parsowania tekstu.
*   **result\_type=Joke:** Definiuje typ danych oczekiwanego wyniku działania agenta jako obiekt klasy `Joke` (zdefiniowanej wcześniej). Oznacza to, że agent ma wyodrębnić informacje z tekstu i umieścić je w odpowiednich polach obiektu `Joke`.
*   **deps\_type=str:** Określa typ danych zależności (dependencies) agenta jako ciąg znaków (`str`). Agent przyjmuje surowy tekst jako dane wejściowe.
*   **system\_prompt=(...):**: Definiuje instrukcję systemową dla agenta. Instrukcja ta określa rolę agenta i format wyjściowy.
    *   Agent ma być parserem żartów, który wyodrębnia komponenty z tekstu wejściowego w celu utworzenia obiektu `Joke`.
    *   Jego jedynym zadaniem jest znalezienie i wyodrębnienie: wprowadzenia (po "Setup:"), puenty (po "Punchline:"), kategorii (po "Category:") oraz procesu myślowego (między tagami `

<think>`).
    *   Wynik ma być zwrócony w określonym formacie, z wyraźnie oznaczonymi polami `setup`, `punchline`, `category` i `thinking`.
    *   Agent *nie powinien* generować nowych treści ani modyfikować tekstu wejściowego.
    *   Odpowiedź agenta powinna zawierać tylko wyodrębnione komponenty, bez dodatkowego tekstu.

Ten kod konfiguruje agenta AI, który będzie analizował tekst żartu i wyodrębniał z niego poszczególne elementy, umieszczając je w obiekcie `Joke`.  Instrukcja systemowa kładzie duży nacisk na dokładność i brak modyfikacji tekstu wejściowego.

In [36]:

joke_judge = Agent(
    model=deepseek_parser_model,
    result_type=JokeAssessment,
    deps_type=str,
    system_prompt=(
        "You are an expert joke critic who analyzes jokes across multiple dimensions. "
        "You will receive a joke in the format 'Setup: <setup> | Punchline: <punchline>'. "
        "Evaluate the joke and return a JokeAssessment with these fields:"
        "- originality (0-10 float)"
        "- humor_level (0-10 float)"
        "- family_friendly (boolean)"
        "- wordplay_quality (0-10 float)"
        "- delivery_structure (0-10 float)"
        "- strengths (list of strings)"
        "- improvement_suggestions (list of strings)"
        "- overall_score (0-10 float)"
        "\nBe constructive but honest in your feedback."
    ),
)


Ten kod tworzy instancję klasy `Agent` z biblioteki `pydantic_ai`, która będzie pełniła rolę sędziego żartów.

*   **joke\_judge = Agent(...):**: Tworzy obiekt o nazwie `joke_judge` będący agentem AI.
*   **model=deepseek\_parser\_model:** Określa, który model językowy ma być używany przez agenta – Qwen2.5 skonfigurowany do parsowania tekstu.
*   **result\_type=JokeAssessment:** Definiuje typ danych oczekiwanego wyniku działania agenta jako obiekt klasy `JokeAssessment` (zdefiniowanej wcześniej). Oznacza to, że agent ma ocenić żart i umieścić wyniki w odpowiednich polach obiektu `JokeAssessment`.
*   **deps\_type=str:** Określa typ danych zależności (dependencies) agenta jako ciąg znaków (`str`). Agent przyjmuje tekst żartu jako dane wejściowe.
*   **system\_prompt=(...):**: Definiuje instrukcję systemową dla agenta. Instrukcja ta określa rolę agenta i format wyjściowy.
    *   Agent ma być ekspertem w krytyce żartów, analizującym je pod różnymi kątami.
    *   Otrzyma żart w formacie "Setup: <setup> | Punchline: <punchline>".
    *   Ma ocenić żart i zwrócić obiekt `JokeAssessment` z określonymi polami: oryginalność, poziom humoru, czy jest odpowiedni dla rodziny, jakość gry słów, struktura przekazu, mocne strony, sugestie dotyczące poprawy oraz ogólna ocena.
    *   Agent ma być konstruktywny i szczery w swoich opiniach.

Ten kod konfiguruje agenta AI, który będzie analizował żarty i dostarczał szczegółową ocenę ich jakości, wykorzystując model Qwen2.5 do przetwarzania tekstu i strukturyzowania wyników w obiekcie `JokeAssessment`.

# Suchar

In [37]:
print("Creating joke...")
joke_text = await joke_creator.run("Create a dad joke")
if not joke_text:
    raise ValueError("Failed to create joke")
print("\nRaw joke text:")
print("-" * 50)
print(joke_text.data)
print("-" * 50)

Creating joke...

Raw joke text:
--------------------------------------------------
<think>
Okay, so the user wants me to create a dad joke following their specific format. First, I need to understand exactly what they're asking for. They provided an example with setup and punchline, and categories like puns, wordplay, etc.

I should focus on family-friendly humor because that's the target audience. Dad jokes usually rely on wordplay or puns, so I'm thinking along those lines. Let me brainstorm a topic—maybe something related to everyday items or animals since those are easy to relate to.

Why don't we ever... oh, wait! Bees might work well. They have a connection with honey and sweetness, which could lead into a pun about being sweetness or desserts.

Putting it together: "Why don’t we ever see bees arguing?" The answer should tie back to the sweet aspect, so something like always agreeing because they’re too busy making honey, implying their "sweet" nature keeps them in harmony.

I t

<ipython-input-37-20e74535158d>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(joke_text.data)


Ten kod generuje żart za pomocą agenta `joke_creator` i wyświetla surowy tekst wygenerowanego żartu.

*   **print("Creating joke...")**: Wyświetla komunikat informujący o rozpoczęciu procesu tworzenia żartu.
*   **joke\_text = await joke\_creator.run("Create a dad joke")**: Uruchamia agenta `joke_creator` z poleceniem "Create a dad joke".  Funkcja `run()` zwraca obiekt zawierający wynik działania agenta. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej (wykonania przez agenta).
*   **if not joke\_text:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to create joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to create joke", co oznacza, że proces tworzenia żartu się nie powiódł.
*   **print("\nRaw joke text:")**: Wyświetla nagłówek przed wyświetleniem surowego tekstu żartu.
*   **print("-" \* 50)**: Wyświetla linię separatora składającą się z 50 znaków "-".
*   **print(joke\_text.data)**: Wyświetla surowy tekst wygenerowanego żartu, który jest dostępny w atrybucie `data` obiektu `joke_text`.
*   **print("-" \* 50)**: Ponownie wyświetla linię separatora składającą się z 50 znaków "-".

W skrócie, kod ten wywołuje agenta do stworzenia żartu, sprawdza czy operacja się powiodła i jeśli tak, to wyświetla surowy tekst wygenerowanego żartu w konsoli.

In [38]:
print("\nParsing joke...")
parsed_joke = await joke_parser.run(joke_text.data)
if not parsed_joke:
    raise ValueError("Failed to parse joke")
print("\nParsed joke text:")
print("-" * 50)
print(parsed_joke.data)
print("-" * 50)



Parsing joke...


<ipython-input-38-11941310037c>:2: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  parsed_joke = await joke_parser.run(joke_text.data)



Parsed joke text:
--------------------------------------------------
setup='Why don’t we ever see bees arguing?' punchline='Because they’re always too busy making honey—agreeable!' category='Puns' thinking='Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a pun. Honey is a funny topic because it’s sweet (literally and figuratively). People often associate bees with honey, so it makes sense to connect their behavior to something delicious.'
--------------------------------------------------


<ipython-input-38-11941310037c>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(parsed_joke.data)


Ten kod parsuje (analizuje) wygenerowany wcześniej żart za pomocą agenta `joke_parser` i wyświetla wynik parsowania.

*   **print("\nParsing joke...")**: Wyświetla komunikat informujący o rozpoczęciu procesu parsowania żartu.
*   **parsed\_joke = await joke\_parser.run(joke\_text.data)**: Uruchamia agenta `joke_parser` z surowym tekstem żartu (`joke_text.data`) jako danymi wejściowymi. Funkcja `run()` zwraca obiekt zawierający wynik parsowania. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej (wykonania przez agenta).
*   **if not parsed\_joke:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to parse joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to parse joke", co oznacza, że proces parsowania żartu się nie powiódł.
*   **print("\nParsed joke text:")**: Wyświetla nagłówek przed wyświetleniem sparsowanego tekstu żartu.
*   **print("-" \* 50)**: Wyświetla linię separatora składającą się z 50 znaków "-".
*   **print(parsed\_joke.data)**: Wyświetla wynik parsowania, który jest dostępny w atrybucie `data` obiektu `parsed_joke`.  Wynik ten powinien być obiektem klasy `Joke`, zawierającym wyodrębnione pola (setup, punchline, category, thinking).
*   **print("-" \* 50)**: Ponownie wyświetla linię separatora składającą się z 50 znaków "-".

W skrócie, kod ten wykorzystuje agenta do przeanalizowania surowego tekstu żartu i wyodrębnienia z niego poszczególnych elementów, a następnie wyświetla wynik parsowania w konsoli.

In [39]:


print("\nAssessing joke...")
assessment = await joke_judge.run(
    f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"
)
if not assessment:
    raise ValueError("Failed to assess joke")
print("\nAssessment:")
print("-" * 50)
print(assessment.data)
print("-" * 50)




Assessing joke...


<ipython-input-39-b574855aeefd>:3: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"



Assessment:
--------------------------------------------------
originality=7.5 humor_level=6.5 family_friendly=True wordplay_quality=3.5 delivery_structure=8.0 strengths=['Clever play on words', 'Brief and concise'] improvement_suggestions=['Increase originality by exploring metaphors in other contexts', 'Experiment with a different setup to subvert expectations before the punchline'] overall_score=6.0
--------------------------------------------------


<ipython-input-39-b574855aeefd>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(assessment.data)


Ten kod ocenia sparsowany żart za pomocą agenta `joke_judge` i wyświetla wynik oceny.

*   **print("\nAssessing joke...")**: Wyświetla komunikat informujący o rozpoczęciu procesu oceny żartu.
*   **assessment = await joke\_judge.run(...)**: Uruchamia agenta `joke_judge` z tekstem żartu sformatowanym jako "Setup: <setup> | Punchline: <punchline>", gdzie `<setup>` i `<punchline>` są pobierane z obiektu `parsed_joke.data`. Funkcja `run()` zwraca obiekt zawierający wynik oceny. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej (wykonania przez agenta).
*   **if not assessment:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to assess joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to assess joke", co oznacza, że proces oceny żartu się nie powiódł.
*   **print("\nAssessment:")**: Wyświetla nagłówek przed wyświetleniem wyniku oceny.
*   **print("-" \* 50)**: Wyświetla linię separatora składającą się z 50 znaków "-".
*   **print(assessment.data)**: Wyświetla wynik oceny, który jest dostępny w atrybucie `data` obiektu `assessment`. Wynik ten powinien być obiektem klasy `JokeAssessment`, zawierającym ocenę oryginalności, poziomu humoru, czy jest odpowiedni dla rodziny i inne kryteria.
*   **print("-" \* 50)**: Ponownie wyświetla linię separatora składającą się z 50 znaków "-".

W skrócie, kod ten wykorzystuje agenta do oceny sparsowanego żartu na podstawie różnych kryteriów i wyświetla wynik oceny w konsoli.

In [40]:
console = Console()

# Create assessment table
table = Table(title="Joke Assessment", box=box.ROUNDED)
table.add_column("Property", style="cyan", width=20)
table.add_column("Value", style="yellow")

# Add joke content
table.add_row("Setup", parsed_joke.data.setup)
table.add_row("Punchline", parsed_joke.data.punchline)
table.add_row("Category", parsed_joke.data.category)
table.add_row("","")

# Add scores
scores = assessment.data
table.add_row("Overall Score", f"[bold]{scores.overall_score:.1f}/10")
table.add_row("", "")
table.add_row("Originality", f"{scores.originality:.1f}/10")
table.add_row("Humor Level", f"{scores.humor_level:.1f}/10")
table.add_row("Wordplay", f"{scores.wordplay_quality:.1f}/10")
table.add_row("Structure", f"{scores.delivery_structure:.1f}/10")
table.add_row(
    "Family Friendly",
    "[green]Yes[/green]" if scores.family_friendly else "[red]No[/red]"
)

# Add strengths
table.add_row("", "")
table.add_row("[bold]Strengths", "")
for strength in scores.strengths:
    table.add_row("", f"• {strength}")

# Add improvement suggestions
table.add_row("", "")
table.add_row("[bold]Improvements", "")
for suggestion in scores.improvement_suggestions:
    table.add_row("", f"• {suggestion}")

# Add thinking process
table.add_row("", "")
table.add_row("[bold]Thinking Process", "")
thinking_lines = parsed_joke.data.thinking.split('\n')
for line in thinking_lines:
    if line.strip():
        table.add_row("", line)

# Print the table
console.print("\n")
console.print(table)


<ipython-input-40-299fc290ecef>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Setup", parsed_joke.data.setup)
<ipython-input-40-299fc290ecef>:10: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Punchline", parsed_joke.data.punchline)
<ipython-input-40-299fc290ecef>:11: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Category", parsed_joke.data.category)
<ipython-input-40-299fc290ecef>:15: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  scores = assessment.data
<ipython-input-40-299fc290ecef>:42: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  thinking_lines = parsed_joke.data.thinking.split('\n')


                                                  Joke Assessment                                                  
╭──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────╮
│ Property             │ Value                                                                                    │
├──────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ Setup                │ Why don’t we ever see bees arguing?                                                      │
│ Punchline            │ Because they’re always too busy making honey—agreeable!                                  │
│ Category             │ Puns                                                                                     │
│                      │                                                                                          │
│ Overall Score        │ 6.0/10                                                                                   │
│                      │                                                                                          │
│ Originality          │ 7.5/10                                                                                   │
│ Humor Level          │ 6.5/10                                                                                   │
│ Wordplay             │ 3.5/10                                                                                   │
│ Structure            │ 8.0/10                                                                                   │
│ Family Friendly      │ Yes                                                                                      │
│                      │                                                                                          │
│ Strengths            │                                                                                          │
│                      │ • Clever play on words                                                                   │
│                      │ • Brief and concise                                                                      │
│                      │                                                                                          │
│ Improvements         │                                                                                          │
│                      │ • Increase originality by exploring metaphors in other contexts                          │
│                      │ • Experiment with a different setup to subvert expectations before the punchline         │
│                      │                                                                                          │
│ Thinking Process     │                                                                                          │
│                      │ Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a │
│                      │ pun. Honey is a funny topic because it’s sweet (literally and figuratively). People      │
│                      │ often associate bees with honey, so it makes sense to connect their behavior to          │
│                      │ something delicious.                                                                     │
╰──────────────────────┴──────────────────────────────────────────────────────────────────────────────────────────╯

Ten kod tworzy i wyświetla tabelę w konsoli, prezentującą szczegółową ocenę wygenerowanego żartu. Wykorzystuje bibliotekę `rich` do formatowania tekstu i tworzenia atrakcyjnej wizualnie tabeli.

*   **console = Console():** Tworzy obiekt `Console` z biblioteki `rich`, który będzie używany do wyświetlania danych w konsoli.
*   **table = Table(title="Joke Assessment", box=box.ROUNDED):** Tworzy obiekt `Table` z tytułem "Joke Assessment" i zaokrąglonymi krawędziami (styl `box.ROUNDED`).
*   **table.add\_column(...):** Dodaje dwie kolumny do tabeli: "Property" (w kolorze cyjanowym, szerokość 20 znaków) oraz "Value" (w kolorze żółtym).
*   **Dodawanie zawartości żartu:** Kod dodaje wiersze do tabeli zawierające treść żartu: setup, punchline i category.
*   **Dodawanie ocen:** Pobiera dane oceny z obiektu `assessment.data` (który jest instancją klasy `JokeAssessment`) i dodaje wiersze do tabeli prezentujące poszczególne kryteria oceny: overall score, originality, humor level, wordplay quality, structure oraz family-friendly status.  Wartości liczbowe są formatowane do jednej cyfry po przecinku (`:.1f`). Status "family-friendly" jest wyświetlany w kolorze zielonym ("Yes") lub czerwonym ("No").
*   **Dodawanie mocnych stron:** Dodaje nagłówek "Strengths" i następnie dodaje wiersze dla każdej z mocnych stron żartu, poprzedzając je kropką.
*   **Dodawanie sugestii poprawy:** Dodaje nagłówek "Improvements" i następnie dodaje wiersze dla każdej z sugestii dotyczących poprawy żartu, poprzedzając je kropką.
*   **Dodawanie procesu myślowego:** Dodaje nagłówek "Thinking Process" i następnie dodaje wiersze zawierające poszczególne linie procesu myślowego agenta (pobrane z `parsed_joke.data.thinking`), dzieląc go na linie za pomocą `\n`.
*   **console.print("\n"):** Dodaje pustą linię przed wyświetleniem tabeli.
*   **console.print(table):** Wyświetla utworzoną tabelę w konsoli, wykorzystując obiekt `Console` do formatowania i kolorowania tekstu.

W skrócie, kod ten tworzy czytelną i estetyczną prezentację oceny żartu w formie tabeli, zawierającą zarówno treść żartu, jak i szczegółową analizę jego jakości oraz sugestie dotyczące poprawy.

# Suchar 2.0

In [41]:
print("1. Creating new joke variation...")
improvement_prompt = (
    f"Create an improved variation of this dad joke using ALL the information below \n" +
    f"Do not change the Setup \n" +
    f"Setup: {parsed_joke.data.setup}\n" +
    f"Punchline: {parsed_joke.data.punchline}\n" +
    f"Category: {parsed_joke.data.category}\n" +
    f"Thinking: {parsed_joke.data.thinking}\n" +
    f"Scores: {scores}"
)

print("Improved prompt:")
print("-" * 50)
print(improvement_prompt)
print("-" * 50)


joke_text = await joke_creator.run(improvement_prompt)

if not joke_text:
    raise ValueError("Failed to create new joke")

print("\nNew joke text:")
print("-" * 50)
print(joke_text.data)
print("-" * 50)


1. Creating new joke variation...
Improved prompt:
--------------------------------------------------
Create an improved variation of this dad joke using ALL the information below 
Do not change the Setup 
Setup: Why don’t we ever see bees arguing?
Punchline: Because they’re always too busy making honey—agreeable!
Category: Puns
Thinking: Thinking: I want to create a light-hearted, family-friendly dad joke using wordplay or a pun. Honey is a funny topic because it’s sweet (literally and figuratively). People often associate bees with honey, so it makes sense to connect their behavior to something delicious.
Scores: originality=7.5 humor_level=6.5 family_friendly=True wordplay_quality=3.5 delivery_structure=8.0 strengths=['Clever play on words', 'Brief and concise'] improvement_suggestions=['Increase originality by exploring metaphors in other contexts', 'Experiment with a different setup to subvert expectations before the punchline'] overall_score=6.0
----------------------------------

<ipython-input-41-af505e6ae695>:5: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup}\n" +
<ipython-input-41-af505e6ae695>:6: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Punchline: {parsed_joke.data.punchline}\n" +
<ipython-input-41-af505e6ae695>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Category: {parsed_joke.data.category}\n" +
<ipython-input-41-af505e6ae695>:8: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Thinking: {parsed_joke.data.thinking}\n" +



New joke text:
--------------------------------------------------
<think>
Alright, so I have this user who wants me to create a joke about bees and honey. The setup is already given: "Why don’t we ever see bees arguing?" And the punchline is "Because they’re always too busy making honey—agreeable!" with the category listed as Puns.

Hmm, first off, I need to understand what makes this joke work. It's a pun on "busy" and "honey," combining them in a clever way. The bee's job is to make honey, which ties into them being agreeable because they're too occupied with their work. That's a nice wordplay.

Now, thinking about family-friendly dad jokes, they usually rely on puns or wordplay that's easy to understand but still makes you laugh. This joke fits well in that category because it's simple and funny without any adult themes.

I should consider if this joke is original enough. Bees making honey is such a common topic, so how can I make it fresh? Maybe play with the "busy" aspect more or

<ipython-input-41-af505e6ae695>:25: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(joke_text.data)


Ten kod generuje nową wersję żartu, uwzględniając informacje z poprzedniej oceny i procesu myślowego.

*   **print("1. Creating new joke variation...")**: Wyświetla komunikat informujący o rozpoczęciu tworzenia ulepszonej wersji żartu.
*   **improvement\_prompt = (...)**: Tworzy ciąg znaków, który będzie używany jako polecenie dla agenta `joke_creator`. Polecenie to zawiera:
    *   Instrukcję stworzenia ulepszonej wersji żartu na podstawie dostarczonych informacji.
    *   Zastrzeżenie, że początek żartu (Setup) ma pozostać niezmieniony.
    *   Treść oryginalnego Setup, Punchline, Category i Thinking z poprzednio wygenerowanego żartu (`parsed_joke.data`).
    *   Wyniki oceny żartu (`scores`), aby agent mógł uwzględnić je w procesie tworzenia nowej wersji.
*   **print("Improved prompt:")**: Wyświetla nagłówek przed wyświetleniem polecenia dla agenta.
*   **print("-" \* 50)**: Wyświetla linię separatora.
*   **print(improvement\_prompt)**: Wyświetla treść polecenia, aby można było sprawdzić, jakie informacje są przekazywane do agenta.
*   **print("-" \* 50)**: Wyświetla kolejną linię separatora.
*   **joke\_text = await joke\_creator.run(improvement\_prompt)**: Uruchamia agenta `joke_creator` z nowym poleceniem (zawierającym informacje o ulepszeniu żartu). Funkcja `run()` zwraca obiekt zawierający wynik działania agenta. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej.
*   **if not joke\_text:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to create new joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to create new joke".
*   **print("\nNew joke text:")**: Wyświetla nagłówek przed wyświetleniem tekstu nowej wersji żartu.
*   **print("-" \* 50)**: Wyświetla linię separatora.
*   **print(joke\_text.data)**: Wyświetla surowy tekst wygenerowanej nowej wersji żartu, który jest dostępny w atrybucie `data` obiektu `joke_text`.
*   **print("-" \* 50)**: Wyświetla kolejną linię separatora.

W skrócie, kod ten tworzy polecenie dla agenta, które instruuje go do wygenerowania ulepszonej wersji żartu na podstawie poprzedniej oceny i procesu myślowego, a następnie wyświetla tekst nowej wersji żartu w konsoli.

In [42]:
print("\nParsing joke...")
parsed_joke = await joke_parser.run(joke_text.data)
if not parsed_joke:
    raise ValueError("Failed to parse joke")
print("\nParsed joke text:")
print("-" * 50)
print(parsed_joke.data)
print("-" * 50)




Parsing joke...


<ipython-input-42-1c6b88054d33>:2: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  parsed_joke = await joke_parser.run(joke_text.data)



Parsed joke text:
--------------------------------------------------
setup='Why don’t we ever see bees arguing?' punchline='Because they’re always too busy making honey—agreeable!' category='Puns' thinking="Alright, so I have this user who wants me to create a joke about bees and honey. The setup is already given: 'Why don’t we ever see bees arguing?' And the punchline is 'Because they’re always too busy making honey—agreeable!' with the category listed as Puns.\n\nHmm, first off, I need to understand what makes this joke work. It's a pun on 'busy' and 'honey,' combining them in a clever way. The bee's job is to make honey, which ties into them being agreeable because they're too occupied with their work. That's a nice wordplay.\n\nNow, thinking about family-friendly dad jokes, they usually rely on puns or wordplay that's easy to understand but still makes you laugh. This joke fits well in that category because it's simple and funny without any adult themes.\n\nThe humor level here is

<ipython-input-42-1c6b88054d33>:7: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(parsed_joke.data)


Ten kod parsuje (analizuje) nowo wygenerowany żart za pomocą agenta `joke_parser` i wyświetla wynik parsowania. Jest to identyczny fragment kodu jak poprzednio, używany do analizy tekstu żartu po jego wygenerowaniu.

*   **print("\nParsing joke...")**: Wyświetla komunikat informujący o rozpoczęciu procesu parsowania żartu.
*   **parsed\_joke = await joke\_parser.run(joke\_text.data)**: Uruchamia agenta `joke_parser` z surowym tekstem nowej wersji żartu (`joke_text.data`) jako danymi wejściowymi. Funkcja `run()` zwraca obiekt zawierający wynik parsowania. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej (wykonania przez agenta).
*   **if not parsed\_joke:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to parse joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to parse joke", co oznacza, że proces parsowania żartu się nie powiódł.
*   **print("\nParsed joke text:")**: Wyświetla nagłówek przed wyświetleniem sparsowanego tekstu żartu.
*   **print("-" \* 50)**: Wyświetla linię separatora składającą się z 50 znaków "-".
*   **print(parsed\_joke.data)**: Wyświetla wynik parsowania, który jest dostępny w atrybucie `data` obiektu `parsed_joke`.  Wynik ten powinien być obiektem klasy `Joke`, zawierającym wyodrębnione pola (setup, punchline, category, thinking).
*   **print("-" \* 50)**: Ponownie wyświetla linię separatora składającą się z 50 znaków "-".

W skrócie, kod ten wykorzystuje agenta do przeanalizowania surowego tekstu nowej wersji żartu i wyodrębnienia z niego poszczególnych elementów, a następnie wyświetla wynik parsowania w konsoli.

In [43]:
print("\nAssessing joke...")
assessment = await joke_judge.run(
    f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"
)
if not assessment:
    raise ValueError("Failed to assess joke")
print("\nAssessment:")
print("-" * 50)
print(assessment.data)
print("-" * 50)



Assessing joke...


<ipython-input-43-2c9fcb1daa0b>:3: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  f"Setup: {parsed_joke.data.setup} | Punchline: {parsed_joke.data.punchline}"



Assessment:
--------------------------------------------------
originality=5.5 humor_level=6.0 family_friendly=True wordplay_quality=7.5 delivery_structure=8.0 strengths=['clever play on words', 'positive outlook'] improvement_suggestions=['Consider refining the punchline for smoother delivery', 'Explore more unique scenarios to increase originality'] overall_score=6.3
--------------------------------------------------


<ipython-input-43-2c9fcb1daa0b>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  print(assessment.data)


Ten kod ocenia sparsowaną nową wersję żartu za pomocą agenta `joke_judge` i wyświetla wynik oceny. Jest to identyczny fragment kodu jak poprzednio, używany do analizy jakości żartu po jego wygenerowaniu i sparsowaniu.

*   **print("\nAssessing joke...")**: Wyświetla komunikat informujący o rozpoczęciu procesu oceny żartu.
*   **assessment = await joke\_judge.run(...)**: Uruchamia agenta `joke_judge` z tekstem żartu sformatowanym jako "Setup: <setup> | Punchline: <punchline>", gdzie `<setup>` i `<punchline>` są pobierane z obiektu `parsed_joke.data`. Funkcja `run()` zwraca obiekt zawierający wynik oceny. Słowo kluczowe `await` oznacza, że program czeka na zakończenie operacji asynchronicznej (wykonania przez agenta).
*   **if not assessment:** Sprawdza, czy zwrócony wynik jest pusty lub nieprawidłowy. Jeśli tak, to...
*   **raise ValueError("Failed to assess joke")**: ...podnosi wyjątek `ValueError` z komunikatem "Failed to assess joke", co oznacza, że proces oceny żartu się nie powiódł.
*   **print("\nAssessment:")**: Wyświetla nagłówek przed wyświetleniem wyniku oceny.
*   **print("-" \* 50)**: Wyświetla linię separatora składającą się z 50 znaków "-".
*   **print(assessment.data)**: Wyświetla wynik oceny, który jest dostępny w atrybucie `data` obiektu `assessment`. Wynik ten powinien być obiektem klasy `JokeAssessment`, zawierającym ocenę oryginalności, poziomu humoru, czy jest odpowiedni dla rodziny i inne kryteria.
*   **print("-" \* 50)**: Ponownie wyświetla linię separatora składającą się z 50 znaków "-".

W skrócie, kod ten wykorzystuje agenta do oceny nowej wersji żartu na podstawie różnych kryteriów i wyświetla wynik oceny w konsoli.

In [44]:

console = Console()

# Create assessment table
table = Table(title="Joke Assessment", box=box.ROUNDED)
table.add_column("Property", style="cyan", width=20)
table.add_column("Value", style="yellow")

# Add joke content
table.add_row("Setup", parsed_joke.data.setup)
table.add_row("Punchline", parsed_joke.data.punchline)
table.add_row("Category", parsed_joke.data.category)
table.add_row("","")

# Add scores
scores = assessment.data
table.add_row("Overall Score", f"[bold]{scores.overall_score:.1f}/10")
table.add_row("", "")
table.add_row("Originality", f"{scores.originality:.1f}/10")
table.add_row("Humor Level", f"{scores.humor_level:.1f}/10")
table.add_row("Wordplay", f"{scores.wordplay_quality:.1f}/10")
table.add_row("Structure", f"{scores.delivery_structure:.1f}/10")
table.add_row(
    "Family Friendly",
    "[green]Yes[/green]" if scores.family_friendly else "[red]No[/red]"
)

# Add strengths
table.add_row("", "")
table.add_row("[bold]Strengths", "")
for strength in scores.strengths:
    table.add_row("", f"• {strength}")

# Add improvement suggestions
table.add_row("", "")
table.add_row("[bold]Improvements", "")
for suggestion in scores.improvement_suggestions:
    table.add_row("", f"• {suggestion}")

# Add thinking process
table.add_row("", "")
table.add_row("[bold]Thinking Process", "")
thinking_lines = parsed_joke.data.thinking.split('\n')
for line in thinking_lines:
    if line.strip():
        table.add_row("", line)

# Print the table
console.print("\n")
console.print(table)


<ipython-input-44-299fc290ecef>:9: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Setup", parsed_joke.data.setup)
<ipython-input-44-299fc290ecef>:10: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Punchline", parsed_joke.data.punchline)
<ipython-input-44-299fc290ecef>:11: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  table.add_row("Category", parsed_joke.data.category)
<ipython-input-44-299fc290ecef>:15: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  scores = assessment.data
<ipython-input-44-299fc290ecef>:42: DeprecationWarning: `result.data` is deprecated, use `result.output` instead.
  thinking_lines = parsed_joke.data.thinking.split('\n')


                                                  Joke Assessment                                                  
╭──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────╮
│ Property             │ Value                                                                                    │
├──────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ Setup                │ Why don’t we ever see bees arguing?                                                      │
│ Punchline            │ Because they’re always too busy making honey—agreeable!                                  │
│ Category             │ Puns                                                                                     │
│                      │                                                                                          │
│ Overall Score        │ 6.3/10                                                                                   │
│                      │                                                                                          │
│ Originality          │ 5.5/10                                                                                   │
│ Humor Level          │ 6.0/10                                                                                   │
│ Wordplay             │ 7.5/10                                                                                   │
│ Structure            │ 8.0/10                                                                                   │
│ Family Friendly      │ Yes                                                                                      │
│                      │                                                                                          │
│ Strengths            │                                                                                          │
│                      │ • clever play on words                                                                   │
│                      │ • positive outlook                                                                       │
│                      │                                                                                          │
│ Improvements         │                                                                                          │
│                      │ • Consider refining the punchline for smoother delivery                                  │
│                      │ • Explore more unique scenarios to increase originality                                  │
│                      │                                                                                          │
│ Thinking Process     │                                                                                          │
│                      │ Alright, so I have this user who wants me to create a joke about bees and honey. The     │
│                      │ setup is already given: 'Why don’t we ever see bees arguing?' And the punchline is       │
│                      │ 'Because they’re always too busy making honey—agreeable!' with the category listed as    │
│                      │ Puns.                                                                                    │
│                      │ Hmm, first off, I need to understand what makes this joke work. It's a pun on 'busy' and │
│                      │ 'honey,' combining them in a clever way. The bee's job is to make honey, which ties into │
│                      │ them being agreeable because they're too occupied with their work. That's a nice         │
│                      │ wordplay.                                                                                │
│                      │ Now, thinking about family-friendly dad jokes, they usually rely on puns or wordplay     │
│                      │ that's easy to understand but s